In [ ]:
import numpy as np
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score
from shutil import copy2
import cv2
from insightface.model_zoo import get_model
import matplotlib.pyplot as plt
from tqdm import tqdm
import shutil
import pickle
import re

In [ ]:
# --- Load model ---
model = get_model("../../weights/sface.onnx", providers=["CUDAExecutionProvider"])
model.prepare(ctx_id=0)

def get_embedding(img_path):
    try:
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError("Image not loaded")
        # Convert BGR → RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        emb = model.get_feat(img_rgb)
        return emb
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return None

In [ ]:
test_dir = "test"
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

for person_name in sorted(os.listdir(test_dir)):
    person_path = os.path.join(test_dir, person_name)
    if not os.path.isdir(person_path):
        continue

    for file_name in sorted(os.listdir(person_path)):
        ext = os.path.splitext(file_name)[1].lower()
        if ext not in image_exts:
            continue

        src = os.path.join(person_path, file_name)
        dst = os.path.join(test_dir, file_name)
        shutil.move(src, dst)

    os.rmdir(person_path)
    print(f"Moved files from {person_name}/ to test/")

In [ ]:
data = np.load("sface_database.npz", allow_pickle=True)
gallery_embeddings = data["embeddings"]
gallery_ids = data["ids"].tolist()

test_dir = "test"
database_dir = "database"
unknown_counter = 1
threshold = 0.42

In [ ]:
def get_unique_filename(directory, file_name):
    base, ext = os.path.splitext(file_name)
    existing_files = set(os.listdir(directory))
    candidate = file_name
    counter = 1
    while candidate in existing_files:
        candidate = f"{base}_{counter:04d}{ext}"
        counter += 1
    return candidate

def get_next_image_filename(directory, prefix):
    existing = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    counts = [int(re.findall(rf"{prefix}_(\d{{4}})\.jpg", f)[0]) for f in existing if re.match(rf"{prefix}_(\d{{4}})\.jpg", f)]
    next_id = max(counts, default=0) + 1
    return f"{prefix}_{next_id:04d}.jpg"

def get_next_unknown_id(database_dir):
    counter = 1
    while os.path.exists(os.path.join(database_dir, f"unknown_{counter}")):
        counter += 1
    return counter

In [ ]:
import time
import psutil
import os
import numpy as np
from tqdm import tqdm
from shutil import copy2
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# ---- system info for memory ----
process = psutil.Process(os.getpid())

# ---- metrics containers ----
latencies_embedding = []          # per-image inference times
latencies_matching = []          # per-image inference times
all_ranked_rel = []
all_ranked_scores = []

results = []
matched_count = 0
unmatched_count = 0

unknown_counter = get_next_unknown_id(database_dir)

start_time = time.time()   # for throughput

for file_name in tqdm(os.listdir(test_dir)):
    test_path = os.path.join(test_dir, file_name)
    expected_id = "_".join(file_name.split("_")[:2])

    # ---------------- MEASURE LATENCY ----------------
    t0 = time.time()
    emb = get_embedding(test_path)
    t1 = time.time()
    latencies_embedding.append((t1 - t0) * 1000.0)   # ms

    if emb is None:
        continue
    query_emb = emb.reshape(1, -1)

    sims = cosine_similarity(query_emb, gallery_embeddings)[0]

    idx = np.argsort(-sims)
    gal_ids_arr = np.array(gallery_ids)
    ranked_ids = gal_ids_arr[idx]
    ranked_scores = sims[idx]
    ranked_rel = (ranked_ids == expected_id).astype(np.int32)

    all_ranked_rel.append(ranked_rel)
    all_ranked_scores.append(ranked_scores)

    best_idx = idx[0]
    best_score = sims[best_idx]

    if best_score >= threshold:
        predicted_id = gallery_ids[best_idx]
    else:
        predicted_id = f"unknown_{unknown_counter}"
        unknown_counter += 1
        gallery_embeddings = np.vstack([gallery_embeddings, query_emb])
        gallery_ids.append(predicted_id)
        np.savez_compressed(
            "sface_database.npz",
            embeddings=gallery_embeddings,
            ids=np.array(gallery_ids)
        )

    match = (predicted_id == expected_id)

    t2 = time.time()
    latencies_matching.append((t2 - t0) * 1000.0)   # ms

    target_dir = os.path.join(database_dir, predicted_id)
    os.makedirs(target_dir, exist_ok=True)

    new_filename = get_next_image_filename(target_dir, predicted_id)
    copy2(test_path, os.path.join(target_dir, new_filename))

    if match:
        matched_count += 1
    else:
        unmatched_count += 1

    results.append({
        "Filename": file_name,
        "Predicted": predicted_id,
        "Expected": expected_id,
        "Score": best_score,
        "Match": match
    })

# ---------------- METRICS CALCULATION ----------------
total_time = time.time() - start_time
avg_latency_embedding = np.mean(latencies_embedding)           # ms per image (calculate embedding)
avg_latency_matching = np.mean(latencies_matching)           # ms per image (matching)
throughput_embedding = len(latencies_embedding) / total_time   # images per second (embedding)
throughput_matching = len(latencies_matching) / total_time   # images per second (matching)
max_memory = process.memory_info().rss / (1024 * 1024)   # MB

print("\n===== PERFORMANCE METRICS =====")
print(f"Processed images   : {len(latencies_embedding)}")
print(f"Average Latency (calculate embedding)   : {avg_latency_embedding:.2f} ms per image")
print(f"Average Latency (identification)   : {avg_latency_matching:.2f} ms per image")
print(f"Throughput (calculate embedding)        : {throughput_embedding:.2f} images/sec")
print(f"Throughput (identification)         : {throughput_matching:.2f} images/sec")
print(f"Peak Memory Usage  : {max_memory:.2f} MB")
print(f"Matched            : {matched_count}")
print(f"Unmatched          : {unmatched_count}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(range(len(latencies_embedding)), latencies_embedding, marker=".", linestyle="-", color="blue")

plt.title(f"Per-image Latency for embedding calculation (SFace, Cosine Similarity, Threshold = {threshold})")
plt.xlabel("Image Index")
plt.ylabel("Latency (ms)")
plt.grid(True)
plt.savefig("per-image-latency-embedding.jpg")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(range(len(latencies_matching)), latencies_matching, marker=".", linestyle="-", color="orange")

plt.title(f"Per-image Latency for face identification (SFace, Cosine Similarity, Threshold = {threshold})")
plt.xlabel("Image Index")
plt.ylabel("Latency (ms)")
plt.grid(True)
plt.savefig("per-image-latency-identification.jpg")
plt.show()

In [ ]:
df = pd.DataFrame(results)
df.to_csv("recognition_results.csv", index=False)
print(df.head())

In [ ]:
labels = ['Matched', 'Unmatched']
sizes = [matched_count, unmatched_count]

plt.figure(figsize=(7.7,5.5))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140)
plt.title(f"Face Recognition Match vs Unmatch (SFace, Cosine Similarity, Threshold = {threshold})")
plt.axis('equal')
plt.savefig('recognition.jpg')
plt.show()

In [ ]:
y_true = df["Expected"]
y_pred = df["Predicted"]
y_bin = df["Match"].astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("Classification Metrics:")
print(f"Accuracy     : {accuracy:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1-score     : {f1:.4f}")

In [ ]:
def cmc_curve(ranked_rel_list, max_rank=50):
    num_probes = len(ranked_rel_list)
    if num_probes == 0:
        return np.zeros(max_rank, dtype=float)

    first_hit_ranks = []
    for rel in ranked_rel_list:
        pos = np.where(rel == 1)[0]
        if len(pos) == 0:
            first_hit_ranks.append(np.inf) 
        else:
            first_hit_ranks.append(pos[0]) 
    first_hit_ranks = np.array(first_hit_ranks)

    cmc = np.zeros(max_rank, dtype=float)
    for k in range(1, max_rank + 1):
        cmc[k-1] = np.mean(first_hit_ranks < k) 
    return cmc

def average_precision_from_rel(ranked_rel):
    rel_total = ranked_rel.sum()
    if rel_total == 0:
        return 0.0
    precisions = []
    hits = 0
    for k, r in enumerate(ranked_rel, start=1):
        if r == 1:
            hits += 1
            precisions.append(hits / k)
    return float(np.mean(precisions))

def mean_average_precision_from_rel(ranked_rel_list):
    if len(ranked_rel_list) == 0:
        return 0.0
    ap_vals = [average_precision_from_rel(rel) for rel in ranked_rel_list]
    return float(np.mean(ap_vals))

max_rank = min(50, len(gallery_ids))
cmc = cmc_curve(all_ranked_rel, max_rank=max_rank)
cmc1 = cmc[0]
cmc3 = cmc[2] if max_rank >= 3 else float('nan')

mAP = mean_average_precision_from_rel(all_ranked_rel)

print("\nRanking Metrics (closed-set style, per-probe ranking):")
print(f"CMC@1 : {cmc1:.4f}")
print(f"CMC@3 : {cmc3:.4f}")
print(f"mAP   : {mAP:.4f}")